# LLM Data Engineering

> 15 trillion tokens -- the training data size of LLaMA 3. The number sounds enormous, but where does it come from? It certainly does not fall from the sky.
>
> This section starts from raw HTML on the internet and walks through the complete data cleaning pipeline: text extraction, quality filtering, deduplication, and data mixing -- why each step is needed and how it is done. We finish by examining why over 70% of post-training data is synthetic.

The starting point of data engineering is Common Crawl -- a non-profit project that crawls the entire web every month, with raw data volume around 500TB/month. This data is in HTML format, mixed with navigation bars, ads, JavaScript code, and comment sections.

Feeding it directly to a model is equivalent to feeding it garbage; we need a systematic method to turn raw HTML into clean, trainable text. The whole process has five steps: text extraction -> quality filtering -> deduplication -> data mixing -> Tokenize. The filtering criteria at each step directly affect the final model quality, so each step needs clear decision rules.

This section breaks down the five steps in order, using real data fragments to show the changes before and after filtering at each step.

## 0. Data Pipeline Overview

First the full picture, then we tackle each part:

```
  Common Crawl (raw HTML, ~500TB/month)
       |
       v
  +-------------+
  | 1. Text Extraction |  HTML -> plain text, remove nav/ads/scripts
  +-------+-----+
         |
         v
  +-------------+
  | 2. Language Filter |  Keep only target languages (English/Chinese etc.)
  +-------+-----+
         |
         v
  +-------------+
  | 3. Quality Filter |  Remove ads/nav/garbage/too-short/too-long
  +-------+-----+
         |
         v
  +-------------+
  | 4. Deduplication   |  Exact dedup + MinHash approximate dedup
  +-------+-----+
         |
         v
  +-------------+
  | 5. Data Mixing     |  Common Crawl + Wiki + Books + Code
  +-------+-----+
         |
         v
    Clean training data
```

## 1. Text Extraction: Pulling Content from HTML

#### 1.1 What is in Common Crawl?

Common Crawl crawls billions of web pages every month, stored in WARC (Web ARChive) format. Each WARC file is a complete HTML page -- containing the main body, plus a lot of content unrelated to it:

- Navigation bar: "Home | About | Contact"
- Ad popups: "Limited time offer! Click to buy!"
- JavaScript code: "function trackUser(){...}"
- CSS styles: ".sidebar { float: right; }"
- Footer: "Copyright 2024. All rights reserved."
- Comment section: "First! Great post! Bump!"

A normal article is wrapped in all these things. The task of text extraction is to identify and keep the main body, and discard the rest. This process is not fully automated -- modern tools (such as trafilatura, resiliparse) combine HTML tag analysis and machine learning models to judge which content is the body.

#### 1.2 Demonstrating the Extraction Process with a Simulated HTML Page

In [ ]:
# === Simulation: A typical Common Crawl HTML page ===
raw_html = """
<!DOCTYPE html>
<html>
<head>
  <title>What is Machine Learning? - AI Blog</title>
  <meta name="description" content="Learn ML basics">
  <script src="tracking.js"></script>
  <style>.ad { color: red; }</style>
</head>
<body>
  <nav>
    <a href="/">Home</a> |
    <a href="/about">About</a> |
    <a href="/contact">Contact</a>
  </nav>

  <div class="sidebar">
    <div class="ad">
      <h3>Sponsored</h3>
      <p>Buy the best AI course! 50% off today only!</p>
      <button>Click Here!</button>
    </div>
    <script type="text/javascript">
      var _gaq = _gaq || [];
      _gaq.push(['_setAccount', 'UA-12345-6']);
      _gaq.push(['_trackPageview']);
    </script>
  </div>

  <article>
    <h1>What is Machine Learning?</h1>
    <p>Machine learning is a subset of artificial intelligence
    that enables systems to learn and improve from experience
    without being explicitly programmed.</p>

    <p>The process of learning begins with observations or data,
    such as examples, direct experience, or instruction.</p>

    <p>Machine learning algorithms build a mathematical model
    based on sample data, known as "training data".</p>
  </article>

  <div class="comments">
    <p>User123: Great article!</p>
    <p>Bot456: Buy followers at cheap-followers.com!</p>
  </div>

  <footer>
    <p>Copyright 2024 AI Blog. All rights reserved.</p>
    <p>Terms of Service | Privacy Policy | Cookie Settings</p>
  </footer>
</body>
</html>
"""

print("=== Raw HTML ===")
print(raw_html[:500])
print("...")
print()
print("^ Mixed inside: nav bar, ads, JS tracking code, comment spam, footer")

In [ ]:
import re
# === Manually simulate the text extraction process ===
print("=== Text Extraction: HTML -> plain text ===")
print()

# Step 1: Remove <script> and <style> tags and their content
def remove_scripts_styles(html):
    """Remove script and style tags (including content)"""
    html = re.sub(r'<script[^>]*>.*?</script>', '', html, flags=re.DOTALL | re.IGNORECASE)
    html = re.sub(r'<style[^>]*>.*?</style>', '', html, flags=re.DOTALL | re.IGNORECASE)
    return html

# Step 2: Remove all HTML tags
def strip_tags(html):
    """Remove all <...> tags"""
    return re.sub(r'<[^>]+>', '', html)

# Step 3: Clean up whitespace
def clean_whitespace(text):
    """Merge extra blank lines, strip leading/trailing whitespace"""
    text = re.sub(r'[ \t]+', ' ', text)  # merge consecutive spaces/tabs
    text = re.sub(r'\n\s*\n\s*\n+', '\n\n', text)  # multiple blank lines -> one
    return text.strip()

# Step by step
print("Step 1 -- Remove script/style tags:")
no_scripts = remove_scripts_styles(raw_html)
print(f"  Original: {len(raw_html)} chars -> After removal: {len(no_scripts)} chars")
print()

print("Step 2 -- Remove all HTML tags:")
plain_text = strip_tags(no_scripts)
print(f"  After removing tags: {len(plain_text)} chars")
print()

print("Step 3 -- Clean up whitespace:")
clean_text = clean_whitespace(plain_text)
print(f"  After cleanup: {len(clean_text)} chars")
print()

print("=" * 60)
print("Extraction result:")
print("=" * 60)
print(clean_text)
print("=" * 60)
print()
print("Note: nav bar (Home|About|Contact) is still there, ads are still there, comments are still there")
print("      -> These need to be removed in the subsequent 'quality filtering' step")

## 2. Quality Filtering

After text extraction, most of the content is still low quality. Imagine what is in Common Crawl: auto-generated SEO pages, gibberish, pages with only a nav bar and no body, the same ad copy repeated a thousand times... If this content is used for training directly, the model learns things worse than learning nothing at all.

So quality filtering is the most critical step in the whole pipeline: it is better to throw away some good data than to let a lot of garbage into the training set. Common methods come in two layers:

#### 2.1 Heuristic Rules (Quickly Filter Obvious Garbage)

These rules do not need a model; they directly inspect statistical features of the text. Each rule targets a common type of garbage:

In [ ]:
import numpy as np
np.random.seed(42)
# === Quality filtering rules: demo one by one ===
print("=== Quality Filtering Rules ===")
print()

# Prepare a few "pending review" texts
samples = [
    {"name": "good article", "text": "Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. The field has grown rapidly since the 2010s, driven by advances in deep learning and the availability of large datasets."},
    {"name": "ad", "text": "BUY NOW!!! Click here!!! Limited time offer!!! Subscribe today and get 50% OFF!!! Don't miss this opportunity!!!"},
    {"name": "table of contents", "text": "Chapter 1. Introduction. Chapter 2. Methods. Chapter 3. Results. Chapter 4. Discussion. Chapter 5. Conclusion. Appendix A. Appendix B. References."},
    {"name": "too short", "text": "Hello world."},
    {"name": "random chars", "text": "asdfjkl; qwerty zxcvbnm @#$%@# 123456789 !!!!!!!"},
    {"name": "repeated template", "text": "This is a blog post.\n" * 30 + "unique content here"},
]

def quality_check(text):
    """Return (keep?, drop reason)"""

    # Rule 1: Length filter
    words = text.split()
    if len(words) < 5:
        return False, f"too short ({len(words)} words < 5)"
    if len(text) > 5000:
        return False, f"too long ({len(text)} chars > 5000)"

    # Rule 2: Average word length -- normal English words are 3-10 chars; gibberish words are very long
    avg_word_len = np.mean([len(w) for w in words])
    if avg_word_len > 12:
        return False, f"abnormal avg word length ({avg_word_len:.1f} > 12)"

    # Rule 3: Special character ratio -- normal text has < 15% punctuation
    special_count = sum(1 for c in text if not c.isalnum() and not c.isspace())
    special_ratio = special_count / max(len(text), 1)
    if special_ratio > 0.25:
        return False, f"too many special chars ({special_ratio:.1%} > 25%)"

    # Rule 4: Line repetition ratio -- template pages have many duplicate lines
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    if len(lines) > 3:
        unique_ratio = len(set(lines)) / len(lines)
        if unique_ratio < 0.4:
            return False, f"line repetition too high ({unique_ratio:.1%} < 40%)"

    # Rule 5: Uppercase letter ratio -- ALL CAPS is usually garbage
    if len(text) > 50:
        upper_ratio = sum(1 for c in text if c.isupper()) / sum(1 for c in text if c.isalpha())
        if upper_ratio > 0.5:
            return False, f"too many uppercase letters ({upper_ratio:.1%} > 50%)"

    return True, "passed"


print(f"{'Text':<20s} {'Result':>14s} {'Reason'}")
print("-" * 65)
for sample in samples:
    passed, reason = quality_check(sample['text'])
    status = "KEEP ✅" if passed else "DROP"
    print(f"{sample['name']:<20s} {status:>14s}  {reason}")

print()
print("Real systems typically have 20-50 such rules.")
print("This filters out about 60-80% of Common Crawl text.")

#### 2.2 Model-Based Filtering -- Hiring a "Language Teacher" to Score

Besides manual rules, you can also use an **already trained lightweight language model** (such as KenLM) to score each article.

The idea is just like doing reading comprehension: a teacher can tell at a glance whether an article "is written by a human" or "is randomly generated".

```
Use a language model to compute the article's "Perplexity" (PPL):
  PPL very low (< 10):   article too simple, like "a a a a a..." -> drop
  PPL normal (10-1000):  looks like normal human writing -> keep
  PPL very high (> 1000): gibberish -> drop
```

Some systems also train a binary classifier: Wikipedia articles = good (positive examples), random Common Crawl articles = bad (negative examples). After training, it scores each article.

**Key insight**: Wikipedia = "textbook-grade" good text; use Wikipedia as a ruler to measure other text.

#### 2.3 PII and Safety Filtering: Not all "normal text" is suitable for training

The rules above filter out gibberish, ads, and template pages. But there is another category of content that looks perfectly normal yet should not be fed directly to a model: personal information and dangerous strings.

PII (Personally Identifiable Information) refers to information that can identify a specific person, such as email addresses, phone numbers, ID numbers, home addresses. If a model repeatedly sees this content during training, it may reproduce it when generating. Safety filtering also checks for API keys, private keys, passwords, malicious scripts, obvious hate/sexual/violent text, etc.

This step is not the same as "whether the article is well written". An article can be very fluent, but contain emails, phone numbers, and keys embedded in it; quality filtering would let it pass, but PII/safety filtering must handle it separately.

Industrial systems typically use two approaches:

| Approach | Suitable Scenario | Example |
|:---|:---|:---|
| Drop entire document | Severe risk hit | Leaked keys, ID numbers, malicious code |
| Redact and keep | Body is valuable, but contains small amounts of PII | Replace email with <EMAIL>, phone with <PHONE> |

The key is not "always drop when you see an email", but first judge the risk level: high risk drops, low risk with valuable body can be redacted.


In [ ]:
import re

# === PII and safety filtering demo ===
print("=== PII / Safety Filtering ===")
print()

records = [
    {
        "name": "normal tutorial",
        "text": "This tutorial explains gradient descent with a small example.",
    },
    {
        "name": "contains email",
        "text": "Contact me at alice@example.com for the dataset notes.",
    },
    {
        "name": "contains phone",
        "text": "My phone number is 13812345678, the training log is here.",
    },
    {
        "name": "suspected key",
        "text": "AWS_SECRET_ACCESS_KEY = ABCDEFGHIJKLMNOPQRSTUVWXYZ1234567890AB",
    },
]

patterns = {
    "EMAIL": r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}",
    "PHONE_CN": r"1[3-9]\d{9}",
    "SECRET": r"(api[_-]?key|secret[_a-z0-9-]*|token|password)\s*[=:]\s*[^\s]+",
}


def redact_or_drop(text):
    """Detect PII / secrets; return the action and the processed text"""
    lowered = text.lower()
    if re.search(patterns["SECRET"], lowered):
        return "drop", "hit suspected secret key"

    cleaned = text
    hit = False
    for label, pattern in patterns.items():
        if label == "SECRET":
            continue
        if re.search(pattern, cleaned):
            hit = True
            cleaned = re.sub(pattern, f"<{label}>", cleaned)

    if hit:
        return "redact & keep", cleaned
    return "keep", cleaned


for record in records:
    action, result = redact_or_drop(record["text"])
    print(f"{record['name']:<20s} -> {action}")
    print(f"  {result}")

print()
print("Key observation: PII/safety filtering addresses leakage risk, not article quality.")


## 3. Deduplication

#### 3.1 Why is Deduplication Important?

A lot of content on the internet is duplicated:
- The same news article is republished by 50 websites (Google News syndication)
- The same code snippet appears in countless blog posts
- The same "Lorem ipsum" placeholder text
- The same cookie notice ("This website uses cookies...")

If we do not deduplicate:
1. The model spends precious training compute learning repeated content
2. Repeated content makes the model "memorize" instead of "understand"
3. Training data statistics become distorted (a passage gets a hundred times its proper weight)

#### 3.2 Two-Layer Deduplication

In [ ]:
import hashlib
# === Exact deduplication ===
print("=== Layer 1: Exact Dedup ===")
print()

# Simulate 5 articles, 2 of which are duplicates
docs = [
    "Machine learning is a subset of artificial intelligence.",
    "Deep learning uses neural networks with many layers.",
    "Machine learning is a subset of artificial intelligence.",  # identical to doc 1
    "Natural language processing deals with text data.",
    "Deep learning uses neural networks with many layers.",  # identical to doc 2
]

print(f"Total {len(docs)} articles")
print()

# Hash deduplication
seen = set()
unique_docs = []

for i, doc in enumerate(docs):
    # SHA256 hash: turn the article into a unique fingerprint
    fingerprint = hashlib.sha256(doc.encode()).hexdigest()[:16]  # show only first 16 chars
    is_new = fingerprint not in seen

    print(f"Doc {i+1}: hash={fingerprint}  {'KEEP ✅' if is_new else 'DUPLICATE, drop'}")

    if is_new:
        seen.add(fingerprint)
        unique_docs.append(doc)

print()
print(f"After dedup: {len(unique_docs)} articles ({len(docs) - len(unique_docs)} removed)")
print()
print("Exact dedup removes about 5-15% of Common Crawl data")
print("But that is not enough -- most duplication is 'paraphrase' rather than 'verbatim copy'")

In [ ]:
import hashlib
# === MinHash approximate deduplication: hand-calculating the principle ===
print("=== Layer 2: MinHash Approximate Dedup ===")
print()
print("Question: 10 billion articles, how to quickly find similar ones?")
print("           Pairwise comparison = 10B^2 = impossible")
print()
print("MinHash idea: compute a 'fingerprint' for each article; similar fingerprints -> similar articles")
print()

# === Manual MinHash demo ===
print("=== MinHash Hand Calculation ===")
print()

# Three articles
doc_A = "the cat sat on the mat and looked at the dog"
doc_B = "the cat sat on the mat and watched the dog"  # differs from A by only one word
doc_C = "quantum mechanics describes behavior of subatomic particles"  # completely different topic

print(f"Doc A: {doc_A}")
print(f"Doc B: {doc_B}")
print(f"Doc C: {doc_C}")
print()

# Step 1: Split each article into an n-gram set (groups of 3 consecutive words)
def get_ngrams(text, n=3):
    words = text.lower().split()
    return set(' '.join(words[i:i+n]) for i in range(len(words) - n + 1))

A_ngrams = get_ngrams(doc_A, 3)
B_ngrams = get_ngrams(doc_B, 3)
C_ngrams = get_ngrams(doc_C, 3)

print(f"Doc A n-grams ({len(A_ngrams)}): {A_ngrams}")
print()
print(f"Doc B n-grams ({len(B_ngrams)}): {B_ngrams}")
print()

# Step 2: Compute Jaccard similarity (intersection/union)
def jaccard(s1, s2):
    inter = len(s1 & s2)
    union = len(s1 | s2)
    return inter / union if union > 0 else 0

j_AB = jaccard(A_ngrams, B_ngrams)
j_AC = jaccard(A_ngrams, C_ngrams)

print(f"A intersect B size: {len(A_ngrams & B_ngrams)}")
print(f"A union B size: {len(A_ngrams | B_ngrams)}")
print(f"Jaccard(A, B) = {len(A_ngrams & B_ngrams)}/{len(A_ngrams | B_ngrams)} = {j_AB:.2%}")
print()
print(f"Jaccard(A, C) = {j_AC:.2%}")
print()

# Step 3: MinHash fingerprint (highly simplified -- simulate with random hashing)
print("=== MinHash Signature Computation ===")
print()

# Simulation: hash each n-gram to an integer, take the smallest K hash values of each article as the signature
def minhash_signature(ngrams, num_hashes=4):
    """
    Generate a MinHash signature for an n-gram set.
    Real MinHash uses random permutations; here we simulate with different seeded hashes.
    """
    sig = []
    for i in range(num_hashes):
        # For each n-gram, compute a hash with seed=i, take the minimum
        min_val = float('inf')
        for ng in ngrams:
            raw = (ng + str(i)).encode()
            h = int(hashlib.sha256(raw).hexdigest(), 16) % 100000
            min_val = min(min_val, h)
        sig.append(min_val) if min_val != float('inf') else sig.append(0)
    return sig

sig_A = minhash_signature(A_ngrams)
sig_B = minhash_signature(B_ngrams)
sig_C = minhash_signature(C_ngrams)

print(f"A MinHash signature: {sig_A}")
print(f"B MinHash signature: {sig_B}")
print(f"C MinHash signature: {sig_C}")
print()

# MinHash similarity = fraction of matching signature positions
def minhash_sim(s1, s2):
    matches = sum(1 for a, b in zip(s1, s2) if a == b)
    return matches / len(s1)

mh_AB = minhash_sim(sig_A, sig_B)
mh_AC = minhash_sim(sig_A, sig_C)

print(f"MinHash approximate similarity A-B: {mh_AB:.2%}  (exact Jaccard: {j_AB:.2%})")
print(f"MinHash approximate similarity A-C: {mh_AC:.2%}  (exact Jaccard: {j_AC:.2%})")
print()
print("-> MinHash turns 'pairwise n-gram comparison' into 'comparing 4 numbers'")
print("-> Orders of magnitude faster! similarity > threshold (e.g. 80%) -> keep only one")

#### 3.3 Decontamination: Benchmark Questions Must Not Leak into the Training Set

Deduplication addresses "internal duplication within the training set". But there is a more subtle problem: benchmark questions have leaked into the training set.

For example, you use HumanEval to evaluate code ability, but the training data already contains HumanEval questions and answers. The model scores will be very high, but this is not strong ability -- it is having seen the answers before the exam. This is called decontamination.

So industrial pipelines usually do two types of checks:

| Check Target | Purpose | Common Methods |
|:---|:---|:---|
| Within training set | Learn less repeated content | exact dedup, MinHash |
| Training set vs benchmark | Prevent benchmark leakage | exact match, n-gram overlap, embedding similarity |

Decontamination is not about making the data "cleaner to look at", but about making the evaluation scores trustworthy.


In [ ]:
# === Decontamination demo: have benchmark questions leaked into the training data ===
print("=== Decontamination: Training set vs Benchmark ===")
print()

train_docs = [
    "To reverse a string in Python, use s[::-1]. This is a common interview trick.",
    "The capital of France is Paris, and it is known for the Eiffel Tower.",
    "Write a function that returns the sum of two numbers. def add(a, b): return a + b",
]

eval_items = [
    "Write a function that returns the sum of two numbers.",
    "What is the capital of France?",
]


def word_ngrams(text, n=6):
    """Split text into consecutive n-word fragments for coarse overlap detection"""
    words = re.findall(r"[a-zA-Z]+", text.lower())
    return set(tuple(words[i:i + n]) for i in range(len(words) - n + 1))


def max_overlap(eval_text, train_texts):
    """Return the max n-gram overlap ratio between an eval question and the training set"""
    eval_grams = word_ngrams(eval_text, n=4)
    best = 0
    best_doc = None
    for doc in train_texts:
        train_grams = word_ngrams(doc, n=4)
        if not eval_grams:
            continue
        overlap = len(eval_grams & train_grams) / len(eval_grams)
        if overlap > best:
            best = overlap
            best_doc = doc
    return best, best_doc


for item in eval_items:
    score, doc = max_overlap(item, train_docs)
    status = "suspected contamination" if score >= 0.6 else "not hit"
    print(f"Eval question: {item}")
    print(f"Max overlap: {score:.0%} -> {status}")
    if doc:
        print(f"Hit document: {doc[:80]}...")
    print()

print("Key observation: decontamination checks overlap between the training set and the benchmark.")


## 4. Data Mixing

Suppose you now have various data sources:

| Source | Volume | Quality | Role |
|:---|:---|:---|:---|
| Common Crawl (filtered) | 10T tokens | Medium | Base knowledge, diversity |
| Wikipedia | 0.1T tokens | Very high | Factual accuracy |
| Books | 0.5T tokens | High | Long-form coherence |
| Code (GitHub) | 1T tokens | Medium | Logical reasoning ability |
| Academic Papers | 0.05T tokens | Very high | Scientific reasoning |

#### 4.1 How to Mix? Not Just Dumping by Raw Volume

Strategy: **high-quality data can be learned multiple times (multiple epochs), low-quality data fewer times.**

In [ ]:
# === Data mixing strategy ===
print("=== Data Mixing: How to Set the Ratio ===")
print()

sources = [
    ("Common Crawl (filtered)", 10000, 0.6, 1),
    ("Wikipedia",               100, 0.95, 4),
    ("Books",                   500, 0.85, 2),
    ("Code (GitHub)",          1000, 0.75, 2),
    ("ArXiv Papers",             50, 0.9,  4),
    ("News",                    300, 0.7,  1),
]

print(f"{'Source':<28s} {'Raw':>10s} {'Quality':>8s} {'Epoch':>6s} {'Effective':>10s} {'Share':>8s}")
print("-" * 78)

total_effective = 0
results = []
for name, size, quality, epochs in sources:
    effective = size * epochs
    total_effective += effective
    results.append((name, size, quality, epochs, effective))

for name, size, quality, epochs, effective in results:
    ratio = effective / total_effective * 100
    print(f"{name:<28s} {size:>6.0f}B   {quality:>7.0%}  {epochs:>4d}x  {effective:>8.0f}B   {ratio:>6.1f}%")

print()
print(f"Total effective data: {total_effective:.0f}B tokens")
print()
print("Key decisions:")
print("  - Wikipedia is only 100B, but epoch=4 -> actually fed 400B (high quality, learn more)")
print("  - Common Crawl is 10T, but epoch=1 -> only learned once (avoid garbage)")
print("  - ArXiv is small but high quality, epoch=4 -> amplifies scientific reasoning")
print()
print("Note: multiple epochs != copying the article 4 times")
print("      Instead, train 4 rounds then re-shuffle -> the model sees a different order each time")

#### 4.2 From "Ratio" to Data Recipe

Above we said "learn high-quality data more times" -- that is the intuition version. In industry it is written as a data recipe: an executable formula that specifies how each type of data is sampled, filtered, and mixed.

Why call it a recipe? Because data is not a one-time wash-and-done. You keep experimenting:

```text
Recipe A: more web, less code
Recipe B: more code and math, less low-quality web
Recipe C: keep only high-scoring web, but reduce total token count
```

Then quickly validate with a small model or proxy training: which recipe has lower loss? Which does better on MMLU, GSM8K, HumanEval? If B is stronger at code but common knowledge drops, that means "too much code, but general text may not be enough".

This is the core of data-centric LLM: you do not only tune the model architecture, you also tune the data recipe.


In [ ]:
# === Data Recipe A/B comparison demo ===
print("=== Data Recipe: Recipe Comparison ===")
print()

recipes = {
    "A_web_heavy": {
        "web": 0.70,
        "wiki": 0.10,
        "books": 0.10,
        "code": 0.05,
        "math": 0.05,
    },
    "B_code_math": {
        "web": 0.45,
        "wiki": 0.10,
        "books": 0.10,
        "code": 0.25,
        "math": 0.10,
    },
    "C_quality_web": {
        "web": 0.50,
        "wiki": 0.20,
        "books": 0.15,
        "code": 0.10,
        "math": 0.05,
    },
}

# These scores are teaching simulations: showing how to compare recipes, not real model results.
eval_scores = {
    "A_web_heavy": {"MMLU": 54, "GSM8K": 28, "HumanEval": 12},
    "B_code_math": {"MMLU": 52, "GSM8K": 39, "HumanEval": 24},
    "C_quality_web": {"MMLU": 57, "GSM8K": 31, "HumanEval": 16},
}

header = (
    f"{'Recipe':<15s} {'web':>5s} {'wiki':>5s} {'books':>6s} "
    f"{'code':>6s} {'math':>6s}  {'MMLU':>6s} {'GSM8K':>6s} {'HEval':>6s}"
)
print(header)
print("-" * len(header))
for name, mix in recipes.items():
    scores = eval_scores[name]
    print(
        f"{name:<15s} "
        f"{mix['web']:>4.0%} {mix['wiki']:>5.0%} {mix['books']:>6.0%} "
        f"{mix['code']:>6.0%} {mix['math']:>6.0%}  "
        f"{scores['MMLU']:>6.1f} {scores['GSM8K']:>6.1f} {scores['HumanEval']:>6.1f}"
    )

print()
print("Key observation: no recipe is absolutely best; it depends on whether you care more about general ability, math, or code.")


#### 4.3 Common Benchmarks: What Ruler Should a Data Recipe Be Evaluated With?

Above we mentioned using MMLU, GSM8K, HumanEval to look at the effect of a data recipe. So what exactly are these names?

First remember a boundary: **benchmarks are usually not pretraining data sources, they are evaluation rulers**. They are like exam papers, used to observe which abilities of the model get stronger or weaker. There are two purposes for caring about benchmarks in training data engineering:

1. Judge the direction of the data recipe. After adding code data, did HumanEval / MBPP go up? After adding math data, did GSM8K / MATH go up? After filtering web, did MMLU / C-Eval drop?
2. Do decontamination checks. Benchmark questions must not leak into the training set, otherwise scores will be inflated.

Common benchmarks can be categorized by ability:

| Ability Type | Common Benchmarks | What It Roughly Tests |
|:---|:---|:---|
| General / subject knowledge | MMLU, MMLU-Pro, C-Eval, CMMLU, AGIEval | Multi-subject exam questions, covering history, law, medicine, STEM, etc. |
| Math reasoning | GSM8K, MATH, AIME | From elementary word problems to competition math, focusing on multi-step reasoning |
| Code ability | HumanEval, HumanEval+, MBPP, LiveCodeBench | Write functions, complete code, pass unit tests |
| Complex reasoning | BBH, GPQA, ARC-Challenge | Hard problems, multi-step reasoning, scientific knowledge and abstract reasoning |
| Common sense / language understanding | HellaSwag, Winogrande, ARC-Easy | Common sense continuation, coreference resolution, basic QA |
| Instruction following / dialogue | IFEval, MT-Bench, AlpacaEval | Whether it answers following the user-required format, constraints, and style |
| Retrieval / RAG | HotpotQA, Natural Questions, RAGAS self-built sets | Whether it can answer based on external materials instead of guessing from memory |
| Agent / tool use | GAIA, SWE-bench, WebArena | Whether it can look up materials, edit code, operate tools and complete tasks |
| Multimodal | MMMU, MMBench, MathVista, VQAv2 | Image QA, chart understanding, visual math reasoning |

A few easily confused names:

- **MMLU / C-Eval / CMMLU / AGIEval**: all lean toward "exam-style knowledge", but with different languages and sources. English models looking only at MMLU is not enough; Chinese models need C-Eval / CMMLU.
- **GSM8K vs MATH**: GSM8K is elementary word problems, many models are near saturation; MATH is harder, closer to competition problems.
- **HumanEval vs MBPP**: both test code, but with different problem sets. HumanEval is more common, MBPP has more problems; industrial reports often look at both together.
- **BBH**: BIG-Bench Hard, picking tasks from the original BIG-Bench where models tend to fail, commonly used to gauge complex reasoning.

So the way to observe a data recipe is not "higher total score is better" -- it is looking at the ability profile:

```text
Stricter web cleaning -> is MMLU / C-Eval more stable?
Higher math data weighting -> do GSM8K / MATH go up?
Higher code data weighting -> do HumanEval / MBPP go up?
Higher instruction data quality -> do IFEval / MT-Bench go up?
```

If a recipe makes HumanEval go up 10 points but MMLU drops 5 points, you have to ask: is this the code-specialized model you wanted, or did code data squeeze out general ability?


In [ ]:
# === Benchmark profile: see which abilities a data recipe change affects ===
print("=== Benchmark Ability Profile: Recipe A vs Recipe B ===")
print()

benchmarks = [
    ("MMLU", "general/subject"),
    ("C-Eval", "Chinese knowledge"),
    ("GSM8K", "math word problems"),
    ("MATH", "competition math"),
    ("HumanEval", "code functions"),
    ("MBPP", "code problems"),
    ("BBH", "complex reasoning"),
    ("IFEval", "instruction following"),
]

# Teaching simulation scores: only used to demo how to read the table, not real model results.
recipe_a = {
    "MMLU": 54,
    "C-Eval": 50,
    "GSM8K": 28,
    "MATH": 12,
    "HumanEval": 12,
    "MBPP": 18,
    "BBH": 36,
    "IFEval": 52,
}
recipe_b = {
    "MMLU": 52,
    "C-Eval": 48,
    "GSM8K": 39,
    "MATH": 20,
    "HumanEval": 24,
    "MBPP": 31,
    "BBH": 39,
    "IFEval": 51,
}

print(f"{'Benchmark':<12s} {'Ability':<20s} {'A':>5s} {'B':>5s} {'Delta':>7s}  Observation")
print("-" * 74)
for name, ability in benchmarks:
    delta = recipe_b[name] - recipe_a[name]
    if delta >= 5:
        note = "clearly stronger"
    elif delta <= -3:
        note = "needs attention"
    else:
        note = "roughly flat"
    print(
        f"{name:<12s} {ability:<20s} "
        f"{recipe_a[name]:>5.1f} {recipe_b[name]:>5.1f} {delta:>+7.1f}  {note}"
    )

print()
print("Key observation: B looks like it added code/math data; code and math went up, but general knowledge dropped slightly.")
print("This is not absolute good or bad; it depends on whether you want a general model or a code/math-specialized one.")


#### 4.4 Common Tools for Running Benchmarks

After knowing benchmarks, the next question is: how to run them? Do not hand-write a bunch of scattered scripts. Industry and research usually use existing evaluation frameworks, because they have already handled prompt, few-shot, metric, parallelism, and result aggregation.

| Tool | Who It Fits | Features | Typical Scenarios |
|:---|:---|:---|:---|
| lm-evaluation-harness (`lm-eval`) | Open-source model researchers | Maintained by EleutherAI, supports many classic NLP/LLM benchmarks, common in paper reproduction | Run MMLU, GSM8K, HellaSwag, BBH, etc. |
| EvalScope (ModelScope) | Domestic model and engineering teams | ModelScope community's one-stop evaluation framework, supports ability eval, performance stress test, result visualization | Run C-Eval, MMLU, GSM8K, also do inference perf testing |
| OpenCompass | People doing systematic model reports | OpenCompass is a large-model evaluation platform, supports 100+ datasets, distributed eval, config-driven experiments | Reproduce model technical reports, build multi-dimensional ability leaderboards |
| OpenAI Evals | People doing custom product eval | OpenAI's eval framework and benchmark registry, good for writing business-private evals | Evaluate API models, do regression testing, custom judges |
| HELM | People doing comprehensive research eval | Stanford CRFM's holistic evaluation framework, emphasizes multi-dimensional and transparent reporting | Research-style horizontal comparison |
| VLMEvalKit / OpenCompass multimodal | People doing VLM | Multimodal evaluation for vision-language models | Run MMMU, MMBench, MathVista |

Three practical judgments:

1. **Want to quickly reproduce experiments**: prioritize `lm-eval` or OpenCompass.
2. **Model is in ModelScope / domestic ecosystem**: EvalScope is more handy.
3. **Want to build your own product test set**: OpenAI Evals or writing your own eval harness layer is more appropriate.

What do the commands look like? Look at the concepts first, no need to run them now:

```bash
# lm-evaluation-harness: evaluate a HuggingFace model
lm_eval --model hf   --model_args pretrained=Qwen/Qwen2.5-0.5B   --tasks mmlu,gsm8k   --num_fewshot 5   --batch_size auto

# EvalScope: run a small eval with an OpenAI-compatible API
evalscope eval   --model your-model-name   --api-url $OPENAI_API_BASE_URL   --api-key $OPENAI_API_KEY   --eval-type openai_api   --datasets gsm8k

# OpenCompass: usually organizes models, datasets, and evaluation methods with config files
python run.py   --models hf_qwen2_5_7b   --datasets mmlu_gen ceval_gen gsm8k_gen
```

When writing a real report, you must simultaneously record: model version, benchmark name, number of few-shots, prompt template, whether CoT, temperature, max tokens, evaluation tool version. Otherwise the same model may produce different scores, and others cannot reproduce.


## 5. Complete Pipeline in Practice: Preparing Data for a 1B Model

The previous four sections covered text extraction, quality filtering, deduplication, and data mixing separately. But in a real project, these stages do not run independently -- they form a pipeline where the output of one step is the input of the next, and a problem in any one stage affects the final training outcome.

Below we string the whole process together and simulate a complete data engineering pipeline: starting from raw Common Crawl WARC files, going through extraction -> filtering -> deduplication -> mixing, and finally producing data that can be sent directly to training. At each step we output statistics, so the reader gets an intuitive sense of "how much was processed, how much was dropped".

The code of a real industrial pipeline is large (usually thousands of lines); here we use a simplified version to show the core logic. The actual effect of each rule can be verified through the output statistics.

In [ ]:
print("=== Practice: 1B LLM Data Pipeline ===")
print()

steps = [
    ("Step 1: Determine data budget", [
        "Chinchilla optimal: N = 1B -> D ~ 20B tokens",
        "Over-training: N = 1B -> D ~ 100B tokens",
        "Choice: 50B tokens (a middle ground, good cost-efficiency)",
    ]),
    ("Step 2: Download Common Crawl", [
        "Download the most recent 2-3 month dumps (about 20TB compressed WARC)",
        "Tools: cc_downloader, HuggingFace datasets",
    ]),
    ("Step 3: Text extraction + language filtering", [
        "WARC -> HTML -> plain text (trafilatura / resiliparse)",
        "Language detection (fastText): keep only English and Chinese",
        "Output: ~2TB plain text (about 400B tokens)",
    ]),
    ("Step 4: Quality filtering", [
        "Heuristic rules: length/word-length/special-chars/line-repetition",
        "KenLM PPL filter: 10 < PPL < 1000",
        "Output: ~200GB (about 40B tokens) -> only 10% remains",
    ]),
    ("Step 5: Deduplication", [
        "Exact dedup: SHA256 hash -> removes ~10%",
        "MinHash approximate dedup: similarity > 80% keep only one -> removes ~20%",
        "Output: ~140GB (about 28B tokens)",
    ]),
    ("Step 6: Mix other sources", [
        "Wikipedia (2x epoch): 4B tokens",
        "Books (2x epoch): 6B tokens",
        "Code GitHub (2x epoch): 10B tokens",
        "Others: 2B tokens",
        "Total: 28B + 22B = 50B tokens",
    ]),
    ("Step 7: Tokenize + packing", [
        "Use a BPE tokenizer to turn text into token IDs",
        "Concatenate into a continuous sequence, cut into 2048/4096 length chunks",
        "Insert <EOS> token at document boundaries",
        "Shuffle + pack into training batches -> start training!",
    ]),
]

for title, details in steps:
    print(title)
    for d in details:
        print(f"  {d}")
    print()

print("Compression ratio summary:")
print("  20TB WARC -> 2TB plain text -> 200GB filtered -> 140GB after dedup")
print("  Final usable data is only ~0.7% of the original download")

## 6. Data Quality > Quantity

This is a repeatedly validated conclusion in the NLP field:

```
T5 paper (2019):
  Cleaned 750GB data trained better > uncleaned 6TB data
  -> 1/8 the data volume, with high enough quality is actually better

Textbooks Are All You Need (2023):
  A 1.3B model trained on 7B high-quality "textbook"-style data
  outperformed larger models trained on more data at code ability
  -> Data quality can partially compensate for parameter count

LLaMA 2 -> LLaMA 3 (2023-2024):
  Model architecture barely changed; the biggest improvements were in data quality and scale
  From 2T -> 15T tokens, data volume grew while quality actually improved
  -> Better filtering, better deduplication, better mixing strategy
```

The practical implication is direct: if you only have limited resources, prioritize investing in improving data quality, not blindly expanding data volume. The training value of 100 good books is far higher than 10,000 spam emails.

## 7. Industrial Cases and Tool Map

Above we hand-wrote a teaching-version pipeline. Does industry do the same? The broad direction is the same, but the engineering form is more systematic.

First look at a few public cases:

| Case | Core Takeaway | What You Should Learn |
|:---|:---|:---|
| LLaMA 3 | Trained on 15T+ tokens of publicly available data, emphasizing data quality and scale | When architecture barely changes, data quality still brings huge gains |
| FineWeb | Built 18T+ English web tokens from Common Crawl, public filtering/dedup experiments | A good dataset is not "download web pages", it is the result of extensive ablation |
| DCLM / DataComp-LM | Fix the training setup, specifically compare different data filtering and mixing strategies | A data recipe can be systematically evaluated like model architecture |
| Dolma | Public 3T token pretraining corpus and data processing report | Open models need open data processing details for reproducibility and analysis |
| RedPajama Data | Provides quality signals, dedup info, and multi-source data | A dataset is not just a pile of text; it can carry quality labels |

Now the tools. You do not need to know how to use them now, but you should know they do not solve the same level of problem:

| Tool | Role | Typical Capabilities |
|:---|:---|:---|
| Data-Juicer | One-stop LLM data processing system | mapper/filter/dedup/selector/evaluator, combined into a data recipe |
| DataTrove | Large-scale text processing pipeline | Common Crawl processing, filtering, deduplication, distributed execution |
| NeMo Curator | NVIDIA's data cleaning and selection tool | Text/image/video data cleaning, PII, safety filtering, deduplication |
| IBM Data Prep Kit | General AI data preparation toolkit | Document conversion, deduplication, PII detection, quality transformation |

Note a common misunderstanding: these libraries do not automatically tell you "what data is best". They provide composable building blocks. The hard part is designing recipes, running experiments, looking at evaluations, and deciding how to change for the next round.

So an industrial pipeline can be understood like this:

```text
Raw data
  -> a chain of operators (extract, filter, redact, dedup, decontaminate, mix)
  -> data recipe
  -> small model trial training / evaluation
  -> adjust recipe based on results
  -> re-process, re-train, re-evaluate
```

This is closer to the real workflow than "write a bunch of cleaning rules".


## 8. From Tokenize to Training Stream

The last step of data engineering is tokenize. What does the data look like after this?

```
Doc A: "Machine learning is great."
  -> tokenize -> [42, 567, 18, 891, 15]

Doc B: "Deep learning is also great."
  -> tokenize -> [123, 567, 18, 456, 891, 15]

Then concatenate into a "token noodle":
[42, 567, 18, 891, 15, <EOS>, 123, 567, 18, 456, 891, 15, <EOS>, ...]

Cut into training chunks (assume seq_len=8):
  Chunk 1: [42, 567, 18, 891, 15, <EOS>, 123, 567]
  Chunk 2: [18, 456, 891, 15, <EOS>, ...]
  ...

Note: chunk boundaries cut through documents!
  The last 3 tokens of Chunk 1 come from Doc B, the first 5 from Doc A
  -> The model sometimes learns "across documents" within a chunk
  -> Solution: add <EOS> to tell the model "a new document starts"
```

#### 8.1 Sequence Packing: Packing Short Documents into Full Chunks

The previous section showed the naive approach of "concatenate -> cut into fixed length". But it has a problem: if document length distribution is uneven, some chunks will be filled with lots of padding, wasting compute.

**Sequence Packing** solves this with a bin-packing algorithm: tightly pack multiple short documents into a fixed-length chunk, leaving as few gaps as possible.

```
Naive approach (with padding waste):
  Chunk 1: [Doc A (500 tokens)][padding (3596 tokens)]  <- 88% wasted
  Chunk 2: [Doc B (800 tokens)][padding (3296 tokens)]  <- 80% wasted

Packing approach (pack as full as possible):
  Chunk 1: [Doc A (500)][Doc B (800)][Doc C (1200)][Doc D (1500)][padding (96)]  <- 98% utilization
```

Real-world gain: training throughput can improve by **2x-4x** (more short documents means more gain).

But Packing has a technical problem that must be solved: **different documents must not attend to each other, otherwise information leaks.** The solution is to add a block-diagonal mask during attention computation.

In [ ]:
# === Sequence Packing hand-calculation demo ===
print("=== Sequence Packing: Bin-Packing Algorithm Hand Calculation ===")
print()

# Simulate the token lengths of 8 documents
docs = [
    ("Doc A", 120),
    ("Doc B", 80),
    ("Doc C", 200),
    ("Doc D", 50),
    ("Doc E", 150),
    ("Doc F", 90),
    ("Doc G", 60),
    ("Doc H", 40),
]

max_seq_len = 256  # fixed chunk length

# First-Fit Decreasing bin packing: sort long docs first, then pack into chunks one by one
sorted_docs = sorted(docs, key=lambda x: -x[1])  # sort by length descending
chunks = []  # each chunk is (doc list, used length)

for name, length in sorted_docs:
    placed = False
    # Try to fit into an existing chunk
    for chunk in chunks:
        if chunk[1] + length + 1 <= max_seq_len:  # +1 for the EOS token
            chunk[0].append((name, length))
            chunk[1] += length + 1
            placed = True
            break
    if not placed:
        chunks.append([[(name, length)], length + 1])

print(f"Total documents: {len(docs)}")
print(f"Chunk capacity: {max_seq_len} tokens")
print(f"Packing result: {len(chunks)} chunks")
print()

total_used = 0
for i, (items, used) in enumerate(chunks):
    doc_names = ' + '.join(name for name, _ in items)
    padding = max_seq_len - used
    util = used / max_seq_len * 100
    total_used += used
    print(f"  Chunk {i+1}: [{doc_names}]")
    print(f"          used {used} tokens, padding {padding}, utilization {util:.0f}%")

overall_util = total_used / (len(chunks) * max_seq_len) * 100
print(f"\nOverall utilization: {overall_util:.0f}%")
print()
print("Compared with the naive approach (one chunk per document):")
naive_chunks = len(docs)
print(f"  Naive: {naive_chunks} chunks")
print(f"  Packing: {len(chunks)} chunks")
print(f"  Saved: {(1 - len(chunks)/naive_chunks)*100:.0f}%")

#### 8.2 Block-Diagonal Attention Mask: Preventing Documents from Seeing Each Other

After packing, a chunk may contain 3-4 different documents. Without any restriction, the model "sees" the content of other documents during attention -- this is information leakage.

Solution: **block-diagonal mask**. Each document can only attend within itself; attention weights between different documents are set to negative infinity (becoming 0 after softmax).

```
Chunk: [Doc A][Doc B][Doc C]

Attention Mask:
        A A A B B C C
    A [ 1 1 1 0 0 0 0 ]   <- A only sees A
    A [ 1 1 1 0 0 0 0 ]
    A [ 1 1 1 0 0 0 0 ]
    B [ 0 0 0 1 1 0 0 ]   <- B only sees B
    B [ 0 0 0 1 1 0 0 ]
    C [ 0 0 0 0 0 1 1 ]   <- C only sees C
    C [ 0 0 0 0 0 1 1 ]

Blocks on the diagonal are each independent = block-diagonal
```

In [ ]:
import numpy as np

# === Block-Diagonal Mask construction demo ===
print("=== Block-Diagonal Attention Mask ===")
print()

# Assume a packed chunk contains 3 documents of lengths 3, 2, 2
doc_lengths = [3, 2, 2]
total_len = sum(doc_lengths)  # 7

# Build a block-diagonal mask (pure numpy)
mask = np.zeros((total_len, total_len))

pos = 0
for length in doc_lengths:
    for i in range(length):
        for j in range(i + 1):
            mask[pos + i, pos + j] = 1.0
    pos += length

print(f"Document lengths: {doc_lengths}, total length: {total_len}")
print()
print("Block-Diagonal Mask (1=visible, 0=masked):")
print("      ", "  ".join([f"t{i}" for i in range(total_len)]))
labels = []
pos = 0
for doc_idx, length in enumerate(doc_lengths):
    for _ in range(length):
        labels.append(f"D{doc_idx}")
print()
for i in range(total_len):
    row = "  ".join([f" {int(mask[i,j])}" for j in range(total_len)])
    print(f"  {labels[i]} t{i}: {row}")

print()
print("Observations:")
print("  D0 (Doc 0) internal: t0->t0, t1->t0,t1, t2->t0,t1,t2  <- causal attention")
print("  D1 (Doc 1) internal: t3->t3, t4->t3,t4               <- fully isolated from D0")
print("  D2 (Doc 2) internal: t5->t5, t6->t5,t6               <- isolated from other docs")
print()
print("In real training you also need:")
print("  1. Each document's position id restarts from 0")
print("  2. Loss is computed only on real tokens; padding does not count")


#### 8.3 Fill-in-the-Middle (FIM): Data Format for Training Code Completion Models

Everything discussed above is the data flow of a general language model: left to right, autoregressively predicting the next token. But for code models, there is another important data construction method.

When writing code, there is code both before and after the cursor. The left is the already-written part (prefix), the right is the not-yet-modified part (suffix). A code completion model needs to understand both sides to fill in the middle accurately. Fill-in-the-Middle (FIM) is the training data format designed for this scenario.

The approach is to randomly split each piece of code into three parts, and rearrange them with special tokens:

```
<PRE> prefix content <SUF> suffix content <MID> middle content
```

During training, after the model sees `<PRE> ... <SUF> ... <MID>`, it must predict the removed middle part. The labels for the prefix and suffix parts are set to ignore_index -- the model can see this context, but no loss is computed for them. Only the middle part computes loss normally.

For example, a 5-line piece of code:

```
Original code:
  a = 1
  b = 2
  c = 3
  d = 4
  e = 5

Split into first 2 + middle 2 + last 1:
  prefix = "a = 1\nb = 2\n"     (model sees, does not predict)
  middle = "c = 3\nd = 4\n"     (training target)
  suffix = "e = 5"              (model sees, does not predict)

FIM format (model input):
  <PRE> a = 1\nb = 2\n <SUF> e = 5 <MID> c = 3\nd = 4\n
  |-- visible, not predicted --|  |- visible -|  |--- to predict ----|
```

Not all samples use FIM. Typically about 50% use the standard autoregressive format (to keep general language understanding and generation ability), and 50% use the FIM format (to learn to look at both sides for code completion). The two formats are mixed in training, so the model can both converse normally and do code completion in an IDE. FIM does not affect the tokenize and packing flow -- it is a text-level rearrangement done before tokenize.

Code models (Code Llama, DeepSeek-Coder, StarCoder, etc.) all treat FIM as a standard configuration. General text models usually do not use it, because writing scenarios are naturally left-to-right.

In [ ]:
import numpy as np

# Hand-calculation demo of FIM format conversion

def fim_transform(code):
    """
    Convert a piece of code to the FIM training format.

    Randomly choose a cut point -> split into prefix/middle/suffix -> rearrange in FIM format
    """
    lines = code.split('\n')
    n = len(lines)

    # Randomly choose cut points
    prefix_len = np.random.randint(1, max(2, int(n * 0.4)))
    middle_len = np.random.randint(1, max(2, int(n * 0.5)))
    prefix_len = min(prefix_len, n - 1)
    middle_len = min(middle_len, n - prefix_len)

    prefix = '\n'.join(lines[:prefix_len])
    middle = '\n'.join(lines[prefix_len:prefix_len + middle_len])
    suffix = '\n'.join(lines[prefix_len + middle_len:])

    fim_text = f"<PRE> {prefix}\n<SUF> {suffix}\n<MID> {middle}"
    return fim_text, prefix, middle, suffix


np.random.seed(42)

sample_code = """def calculate(x, y):
    z = x + y
    result = z * 2
    if result > 100:
        return result
    else:
        return 0

print(calculate(3, 4))"""

fim_text, prefix, middle, suffix = fim_transform(sample_code)

print("=== FIM Format Conversion Hand Calculation ===")
print()
print("Original code:")
print(sample_code)
print()
print("--- After splitting ---")
print(f"Prefix ({len(prefix)} chars, model sees):")
print(prefix)
print()
print(f"Middle ({len(middle)} chars, training target):")
print(middle)
print()
print(f"Suffix ({len(suffix)} chars, model sees):")
print(suffix if suffix else '(empty)')
print()
print("--- FIM format (model input) ---")
print(fim_text)
print()

print("=== Label Mask During Training ===")
print()
print("Rules:")
print("  <PRE> ... <SUF> ... <MID> these special tokens -> known but not predicted")
print("  prefix and suffix content        -> label = ignore_index")
print("  middle content                   -> label = normal token (participates in loss)")
print()
print("Key observations:")
print("1. FIM rearranges text so the model can leverage bidirectional context during training")
print("2. <PRE> and <SUF> provide front/back context; after <MID> the model predicts the removed part")
print("3. Only the loss in the middle region is kept -- prefix/suffix give info but no penalty")
print("4. About 50% samples use standard format + 50% use FIM -> balance autoregressive generation and code completion")

## 9. Post-Training Data: From Real Annotations to Synthetic Data

The previous 8 sections were all about pretraining data -- scraping HTML from the internet and cleaning it into token noodles. But LLM training has another stage: post-training (SFT, RLHF, DPO) does not need token noodles, it needs structured data like "instruction-response" pairs and "preference pairs".

Here is a core economic problem: high-quality real annotations are too expensive.

| Data Type | Per-Item Cost (USD) | A Qualified Dataset |
|:---|:---|:---|
| Ordinary instruction annotation | 3 - 5 | 50k items ~= $200k |
| Expert-level code/math annotation | 10 - 30 | 10k items ~= $200k |
| Preference pairs (A vs B) | 5 - 10 | 100k pairs ~= $1M |

The LIMA paper trained on 1000 carefully annotated items, costing close to $10k; InstructGPT's preference data collection cost over a million dollars.

This cost cannot sustain the weekly iteration speed of modern LLMs. So almost all major labs have turned to **synthetic data**: letting strong models (GPT-4, Claude, Qwen-72B) generate instructions, responses, and preference pairs, then using this data to train the target model.

DeepSeek-R1's technical report shows that synthetic data accounts for over 70% of its post-training data; Qwen2.5's technical report also mentions large-scale synthetic data. This section looks at four mainstream synthetic methods.

### 9.1 Self-Instruct: The Model Sets Its Own Questions

Stanford's 2022 Self-Instruct paper first systematically proposed: treat the model as an instruction generator.

The process has four steps:

1. **Seed pool**: humans write 100-200 instructions as seeds (covering different task types)
2. **Generation**: sample a few from the seed pool and let the model generate new instructions
3. **Response**: let the model write responses to its own new instructions
4. **Filtering**: use rules (dedup, low-quality filtering) + model scoring, keep high-quality samples and add them to the seed pool

Repeat steps 2-4, and the seed pool grows exponentially. The original paper expanded from 175 seeds to 52K instructions, costing less than $500.

Below we simulate this process in Python, making the "generate-filter" loop explicit.

In [ ]:
# === Self-Instruct process simulation ===
# Demo: from 3 seed instructions, run two rounds of self-generation, observe seed pool growth

import random
random.seed(42)

# Step 1: human seed instructions (100-200 in real scenarios)
seed_pool = [
    "Translate this sentence into English: the weather is nice today",
    "Summarize the main points of the following passage: ...",
    "Explain what gravity is to a 5-year-old",
]

# Step 2: define the simulation of "model generation"
# In real scenarios call GPT-4 / Claude; here we simulate with preset results
def mock_generate_instructions(seeds, n=3):
    """Simulate: model sees seeds, generates new instructions"""
    templates = [
        "Rewrite this passage in a {style} style: ...",
        "Rewrite the following in {lang}: ...",
        "Explain {concept} to an {age}-year-old: ...",
        "Summarize this article about {topic}: ...",
        "Compare the similarities and differences of {a} and {b}: ...",
    ]
    fills = [
        {"style": "formal"}, {"style": "colloquial"}, {"lang": "French"},
        {"age": "8"}, {"age": "high schooler"}, {"concept": "photosynthesis"},
        {"concept": "quantum mechanics"}, {"topic": "climate change"}, {"a": "Python", "b": "Go"},
    ]
    new = []
    for _ in range(n):
        t = random.choice(templates)
        f = random.choice(fills)
        try:
            new.append(t.format(**f))
        except KeyError:
            new.append(t)
    return new

# Step 3: filtering (dedup + length + keywords)
def filter_instructions(candidates, existing):
    kept = []
    seen = set(existing)
    for inst in candidates:
        if inst in seen:
            continue
        if len(inst) < 5:
            continue
        kept.append(inst)
        seen.add(inst)
    return kept

# Run two rounds
for round_id in range(1, 3):
    seeds_sample = random.sample(seed_pool, k=min(3, len(seed_pool)))
    new_insts = mock_generate_instructions(seeds_sample, n=5)
    kept = filter_instructions(new_insts, seed_pool)
    seed_pool.extend(kept)
    print(f"=== Round {round_id} ===")
    print(f"  Generated this round: {len(new_insts)} items")
    print(f"  Kept after filtering: {len(kept)} items")
    print(f"  Seed pool total: {len(seed_pool)} items")
    for inst in kept:
        print(f"    + {inst}")
    print()

print("=" * 60)
print("Key observations:")
print(f"  Seed pool went from 3 -> {len(seed_pool)} items (two rounds)")
print(f"  Real Self-Instruct runs 10+ rounds, from 175 -> 52K items")
print(f"  Filtering determines quality: dedup avoids mode collapse into a few templates")

### 9.2 Evol-Instruct: Letting Instructions Evolve

Self-Instruct has a problem: the new instructions are "too similar" to the seeds, and the task difficulty distribution is narrow. The WizardLM paper proposed Evol-Instruct -- instead of generating from scratch, it **evolves** existing instructions.

Three directions of evolution:

- **In-depth evolution**: make the instruction more complex. Example: `compute 2+2` -> `compute (3.14 x 7 + 2) / 0.5 and explain each step`
- **In-breadth evolution**: change the topic but keep the difficulty. Example: `write a Python sort function` -> `write a Python dedup function`
- **Eliminating**: filter out failed generated instructions

This systematically covers the full "easy-medium-hard" difficulty distribution.

In [ ]:
# === Evol-Instruct demo ===
# Run 3 rounds of in-depth evolution on the same seed instruction

seed_instruction = "Write a function that takes two numbers and returns the larger one"

# Prompt templates simulating in-depth evolution (fed to GPT-4 in real scenarios)
evolution_prompts = [
    """Make the following instruction more complex, adding 1-2 constraints.
    Instruction: {inst}
    New instruction:""",
    """Rewrite the following instruction into a more professional version, requiring boundary condition handling.
    Instruction: {inst}
    New instruction:""",
    """Extend the following instruction into a multi-step task, adding exception handling requirements.
    Instruction: {inst}
    New instruction:""",
]

# Simulated evolution results (LLM output in real scenarios)
evolved_chain = [
    seed_instruction,
    "Write a function that takes two floats and returns the larger one; require handling of NaN",
    "Write a Python function max_of_two(a, b) that handles floats, NaN, Inf, "
    "with type annotations and docstring; require passing mypy static checks",
    "Implement max_of_two(a, b), supporting mixed comparison of int/float/Decimal types; "
    "raise a custom TypeError; write a complete pytest test suite covering boundary conditions",
]

print("=== Evol-Instruct In-Depth Evolution Chain ===\n")
for i, inst in enumerate(evolved_chain):
    if i == 0:
        print(f"[seed]   {inst}")
    else:
        print(f"[gen {i}] {inst}")
    print()

print("=" * 60)
print("Key observations:")
print("  In-depth evolution drives the instruction deeper on the 'same topic'")
print("  After 3 generations, it went from a 1-line code task to a 30-line engineering task")
print("  WizardLM used this method to evolve 250 seeds into 250K high-quality instructions")

### 9.3 Distillation from Strong Models: Legitimate "Reverse Engineering"

The most direct synthetic data method: **use a strong model as a teacher to create training data for a small model**.

Process:

1. Prepare a batch of prompts (from real user logs, open-source instruction sets, Self-Instruct generation)
2. Call the strong model API (GPT-4 / Claude / Qwen-72B) to generate responses
3. Use (prompt, strong model response) as training pairs
4. Train the small model with LoRA or full fine-tuning

This is the method used by Alpaca (Stanford), Vicuna, and the Qwen2.5 series. **Alpaca spent only $600 on API calls** to generate 52K training items, and the trained model approached GPT-3.5 level.

The key constraint is **compliance**: most API ToS forbid using outputs to train competing models. Real industrial practice usually uses the company's own strong model as the teacher.

In [ ]:
# === Distillation data generation template ===
# Demo: call an OpenAI-compatible API to batch-turn a prompt list into training data

# Real run requires: pip install openai and setting an API key
# Below only shows the data flow, no actual call

def distill_from_strong_model(prompts, model_name="qwen-plus",
                              api_key="sk-xxx", base_url=None):
    """
    Use a strong model to generate responses for a batch of prompts, return SFT training pairs.
    Real scenarios add: concurrency, retries, length filtering, diversity filtering.
    """
    from openai import OpenAI
    client = OpenAI(api_key=api_key, base_url=base_url)

    distilled = []
    for prompt in prompts:
        # Real scenarios wrap a system prompt with a chat template
        resp = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": prompt},
            ],
            temperature=0.7,  # diversity
            max_tokens=1024,
        )
        answer = resp.choices[0].message.content
        distilled.append({"prompt": prompt, "response": answer})
    return distilled


# Simulated data: 3 prompts, showing the training-pair format
demo_prompts = [
    "Implement an LRU cache in Python",
    "Explain the principle of batch normalization",
    "Translate 'I'd like to book a ticket to Beijing' into English",
]

# Simulated strong-model output (returned by the API in real scenarios)
mock_distilled = [
    {"prompt": demo_prompts[0],
     "response": "from collections import OrderedDict\nclass LRUCache: ..."},
    {"prompt": demo_prompts[1],
     "response": "Batch Norm normalizes each layer's input along the batch dimension..."},
    {"prompt": demo_prompts[2],
     "response": "I'd like to book a ticket to Beijing."},
]

print("=== Distilled SFT Training Pairs ===\n")
for i, pair in enumerate(mock_distilled, 1):
    print(f"--- Training pair #{i} ---")
    print(f"prompt:   {pair['prompt']}")
    print(f"response: {pair['response'][:80]}...")
    print()

print("=" * 60)
print("Key observations:")
print("  Distillation cost ~= API call cost (Alpaca 52K items ~= $600)")
print("  The bottleneck in real scenarios is not money, it is prompt pool diversity")
print("  Industrial practice: mix real user logs + Self-Instruct + Evol")

### 9.4 STaR: Self-Improvement of Reasoning Data

The previous three methods are all "teacher sets the question, teacher answers". STaR (Self-Taught Reasoner, Zelikman 2022) goes further: **the model answers for itself, and only the reasoning traces of correct answers are kept as training data**.

Process:

1. Give the model a question with a standard answer (math, code, logic)
2. Let the model generate a reasoning process + answer
3. If the answer is correct: add the reasoning process to the training set
4. If the answer is wrong: give a hint (usually the correct answer), let the model reason again (rationalization)
5. Retrain the model with the expanded training set
6. Repeat for multiple rounds, the model gets stronger each round

Core insight: **the reasoning process of correct answers is a high-quality self-teaching sample** -- it both fits the model's current ability and points to the correct answer.

In [ ]:
# === STaR process simulation ===
# Demo: run one round of STaR, observe how the training set expands

import random
random.seed(42)

# Prepare a question bank (with standard answers)
problems = [
    {"q": "3 x 7 + 2 = ?", "a": "23"},
    {"q": "If 2x = 10, x = ?", "a": "5"},
    {"q": "sqrt(81) = ?", "a": "9"},
    {"q": "How many times is 12 of 3?", "a": "4"},
    {"q": "(5 + 3) x 2 = ?", "a": "16"},
]

# Simulate the current model's "reasoning + answering" ability (accuracy about 60%)
def mock_model_generate(problem):
    q = problem["q"]
    correct_a = problem["a"]
    # 60% chance of correct answer
    if random.random() < 0.6:
        return {"reasoning": f"derive {q} -> got {correct_a}", "answer": correct_a, "correct": True}
    else:
        wrong = str(int(correct_a) + random.choice([-2, -1, 1, 2]))
        return {"reasoning": f"derive {q} -> miscalculated -> {wrong}", "answer": wrong, "correct": False}

# Run one round of STaR
training_set = []
print("=== STaR One Iteration ===\n")
for p in problems:
    out = mock_model_generate(p)
    status = "include" if out["correct"] else "drop"
    print(f"Question: {p['q']}")
    print(f"  Reasoning: {out['reasoning']}")
    print(f"  Answer: {out['answer']}  Correct answer: {p['a']}  {status}")
    if out["correct"]:
        training_set.append({
            "prompt": p["q"],
            "reasoning": out["reasoning"],
            "answer": out["answer"],
        })

print(f"\n{'=' * 60}")
print(f"Round result:")
print(f"  Total questions: {len(problems)}")
print(f"  Training pairs included: {len(training_set)} ({len(training_set)/len(problems)*100:.0f}%)")
print(f"\nKey observations:")
print(f"  STaR solidifies the reasoning process the model 'can answer correctly' into training data")
print(f"  Wrong samples are dropped, not polluting the training set")
print(f"  After multiple iterations, the model's reasoning ability keeps improving (math accuracy 30% -> 50%+ in the paper)")

### 9.5 Data Flywheel: Train -> Collect -> Retrain

Stringing the above four methods together gives a **data flywheel**:

```text
            +-----------------------------+
            v                             |
  +-- Train v_n model --+                 |
  |                      v                |
  |   Generate data with v_n              |
  |   |- Self-Instruct generates new instructions
  |   |- Evol-Instruct evolves instructions
  |   |- STaR collects correct reasoning
  |   |- Distill from a stronger v_n side branch
  |                      v                |
  |   Filter + dedup + quality scoring    |
  |                      v                |
  |   Train v_{n+1} model ----------------+
  |
  +-- Each round, model stronger -> generated data better -> next round model stronger
```

Key risks of the flywheel:

- **Mode collapse**: the model only generates samples it is good at, the training set gets narrower. Need Evol-Instruct to actively expand difficulty
- **Error amplification**: if the model is 60% correct, the 60% kept by STaR may still have systematic bias. Need a verifier or reward model as a backstop
- **Distribution drift**: each round changes the training data distribution, which may cause catastrophic forgetting

Industrial practice (Qwen2.5, DeepSeek-V3) usually uses an **AI + human** mix: AI generates the first version, humans spot-check 5%-10% for quality assessment, find systematic problems, adjust the generation prompt, then run the next round.

### 9.6 Red Lines of Synthetic Data

Synthetic data is not a panacea; known pitfalls:

- **Compliance issues**: using GPT-4 outputs to train a competing model violates OpenAI ToS. You can only use your own strong models
- **Insufficient diversity**: LLMs tend to generate "responses near the median", training data loses long-tail coverage
- **Real user data is still gold**: the performance of models trained on synthetic data on real distributions must be validated with real data
- **Benchmark contamination**: synthetic instructions may unintentionally resemble the benchmark, inflating scores

The core consensus across LIMA, Alpaca, WizardLM, Self-Rewarding LM, STaR and related papers: **a small amount of real high-quality data > a large amount of low-quality synthetic data > a large amount of low-quality real data**. The value of synthetic data is to "supplement", not to "replace".

## Summary

Confirm you understand these (check in order):

1. ✅ Data sources: Common Crawl (main body) + Wikipedia + Books + Code + Papers
2. ✅ Pipeline five steps: text extraction -> language filtering -> quality filtering -> deduplication -> data mixing
3. ✅ HTML -> Text: remove script/style tags + remove HTML tags + clean whitespace
4. ✅ Quality filtering: heuristic rules (length/word-length/symbols) + PPL (language model scoring)
5. ✅ PII/safety filtering: text with normal quality may still need redaction or dropping due to emails, phone numbers, keys
6. ✅ Exact dedup: SHA256 hash, keep only one copy of completely identical items
7. ✅ MinHash: turn articles into fingerprints (n-gram -> hash -> take smallest K), similar fingerprints = similar articles
8. ✅ Decontamination: the training set and benchmark must not highly overlap, otherwise benchmark scores are untrustworthy
9. ✅ Data mixing: upgrade from "high quality more epochs" to an evaluable data recipe
10. ✅ Common benchmarks: MMLU/C-Eval for knowledge, GSM8K/MATH for math, HumanEval/MBPP for code, BBH/GPQA for hard-problem reasoning
11. ✅ Evaluation tools: lm-eval, EvalScope, OpenCompass, OpenAI Evals are responsible for running benchmarks reproducibly
12. ✅ Industrial tools: Data-Juicer, DataTrove, NeMo Curator, IBM Data Prep Kit all help you combine and run data operators
13. ✅ After Tokenize: concatenate into a "token noodle" -> cut into fixed-length chunks -> start training
14. ✅ FIM (Fill-in-the-Middle): rearrange text with <PRE>/<SUF>/<MID> to train code completion ability

15. ✅ Post-training data is 70%+ synthetic: real annotation cost cannot sustain iteration speed
16. ✅ Self-Instruct: 175 seeds -> 52K instructions, cost < $500
17. ✅ Evol-Instruct: three directions (depth/breadth/eliminate) systematically expand the difficulty distribution
18. ✅ Distillation: Alpaca spent $600 on API calls to approach GPT-3.5; the compliance constraint is "cannot use competing APIs"
19. ✅ STaR: only keep reasoning traces of correct answers as training data, self-improving over multiple iterations
20. ✅ Data flywheel risks: mode collapse, error amplification, distribution drift; AI + human mix is the industrial mainstream

**One-line summary**: data engineering is one of the most critical stages of LLM training. The teaching-version pipeline helps you see the main thread; the key of the industrial-version pipeline is connecting cleaning, redaction, deduplication, decontamination, mixing, and evaluation into a closed loop, and iterating on the data recipe continuously.

## Exercises

The 3 small exercises below are all "fill in the blank + assert check". Try to think for 3 minutes first; you can ask AI for ideas, break down steps, or check direction, but it is not recommended to have AI produce the complete answer directly.


In [ ]:
# Exercise 1: Complete an email redaction function
# Hint: you can use re.sub to replace matched emails with <EMAIL>.

import re

def redact_email(text):
    """Replace emails in the text with <EMAIL>"""
    email_pattern = r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"
    # TODO: change the None below to a single line of re.sub(...)
    result = None
    return result

# After filling in, uncomment the next line and run the check
# assert redact_email("mail me at a@b.com") == "mail me at <EMAIL>"
# print("Exercise 1 passed: you can do the most basic PII redaction.")


In [ ]:
# Exercise 2: Complete Jaccard similarity
# Hint: intersection uses &, union uses |.

def jaccard_similarity(a, b):
    """Compute the Jaccard similarity of two sets"""
    # TODO: change the None below to len(intersection) / len(union)
    score = None
    return score

# After filling in, uncomment the three lines below and run the check
# s1 = {"the cat", "cat sat", "sat down"}
# s2 = {"the cat", "cat sat", "sat here"}
# assert abs(jaccard_similarity(s1, s2) - 0.5) < 1e-9
# print("Exercise 2 passed: you can compute the core similarity of approximate dedup.")


In [ ]:
# Exercise 3: Complete the weighted sampling amount of a data recipe
# Hint: effective sampling amount = raw token count * sampling weight.

sources = [
    {"name": "web", "tokens": 1000, "weight": 0.5},
    {"name": "wiki", "tokens": 100, "weight": 2.0},
    {"name": "code", "tokens": 200, "weight": 1.5},
]

# TODO: change None to a list comprehension that computes the effective sampling amount for each source
sampled_tokens = None

# After filling in, uncomment the two lines below and run the check
# assert sampled_tokens == [500, 200, 300]
# print("Exercise 3 passed: you understand sampling weights in a data recipe.")
